# 🔥 Smoke & Fire Detection — YOLOv8 Training on Google Colab

**Dataset:** D-Fire (sayedgamal99/smoke-fire-detection-yolo)  
**Model:** YOLOv8n (lightweight, fast)  
**Classes:** `fire` (0), `smoke` (1)  

> **Before running:** Go to `Runtime → Change runtime type` and select **GPU (T4)**.

## 1. Check GPU & Environment

In [ ]:
import os

# Verify GPU is available
!nvidia-smi

# Working directory — all outputs go here
OUTPUT_PATH = "/content/fire-smoke"
DATASET_PATH = os.path.join(OUTPUT_PATH, "dataset")
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(DATASET_PATH, exist_ok=True)

print(f"\nDataset path: {DATASET_PATH}")
print(f"Output path:  {OUTPUT_PATH}")

Fri Mar 27 16:23:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Dependencies

In [ ]:
!pip install ultralytics kaggle -q

## 3. Download Dataset from Kaggle

You need a Kaggle API key (`kaggle.json`).  
Get it from **kaggle.com → Your Profile → Settings → API → Create New Token**.  
Then upload it when prompted below.

In [ ]:
from google.colab import files
import os, json

# Upload your kaggle.json
print("Upload your kaggle.json file:")
uploaded = files.upload()

# Install the credentials
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
kaggle_json_path = os.path.expanduser("~/.kaggle/kaggle.json")
with open(kaggle_json_path, "w") as f:
    f.write(list(uploaded.values())[0].decode())
os.chmod(kaggle_json_path, 0o600)
print("kaggle.json installed ✓")

Upload your kaggle.json file:


Saving kaggle.json to kaggle (3).json
kaggle.json installed ✓


In [ ]:
# Download and unzip the dataset
!kaggle datasets download -d sayedgamal99/smoke-fire-detection-yolo -p {DATASET_PATH} --unzip
print("\nDataset contents:")
!ls {DATASET_PATH}

Dataset URL: https://www.kaggle.com/datasets/sayedgamal99/smoke-fire-detection-yolo
License(s): CC0-1.0
100% 2.84G/2.84G [02:49<00:00, 18.0MB/s]


Dataset contents:
data  data.yaml


## 4. Explore Dataset Structure

In [ ]:
import os

for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(DATASET_PATH, split, 'images')
    lbl_dir = os.path.join(DATASET_PATH, split, 'labels')
    if os.path.exists(img_dir):
        n_imgs = len(os.listdir(img_dir))
        n_lbls = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
        print(f"{split:>6}: {n_imgs:>5} images | {n_lbls:>5} labels")

# Preview data.yaml
yaml_path = os.path.join(DATASET_PATH, 'data.yaml')
if os.path.exists(yaml_path):
    print("\n--- data.yaml ---")
    with open(yaml_path) as f:
        print(f.read())


--- data.yaml ---
path: /kaggle/working/D Fire Dataset  # dataset root dir
train: data/train/images  # train images (relative to 'path')
val: data/val/images  # val images (relative to 'path')
test: data/test/images  # test images (relative to 'path')

# Classes
names: ['smoke', 'fire']  # Replace with your actual class names

# Counts
nc: 2  # number of classes
train_count: 14122
val_count: 3099
test_count: 4306



## 5. Patch data.yaml for Colab Paths

In [ ]:
import os

for root, dirs, files in os.walk("/content/fire-smoke/dataset"):
    level = root.replace("/content/fire-smoke/dataset", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files:
            print(f"{indent}  {f}")



dataset/
  data.yaml
  data/
    train/
      images/
      labels/
    test/
      images/
      labels/
    val/
      images/
      labels/


In [ ]:
import yaml

DATASET_PATH = "/content/fire-smoke/dataset/data"

yaml_src = "/content/fire-smoke/dataset/data.yaml"
yaml_dst = os.path.join(OUTPUT_PATH, 'data.yaml')

with open(yaml_src) as f:
    cfg = yaml.safe_load(f)

cfg['path']  = DATASET_PATH
cfg['train'] = 'train/images'
cfg['val']   = 'val/images'
cfg['test']  = 'test/images'

print("Classes:", cfg.get('names'))
print("NC:", cfg.get('nc'))

with open(yaml_dst, 'w') as f:
    yaml.dump(cfg, f)

print(f"\nPatched data.yaml written to: {yaml_dst}")
with open(yaml_dst) as f:
    print(f.read())

Classes: ['smoke', 'fire']
NC: 2

Patched data.yaml written to: /content/fire-smoke/data.yaml
names:
- smoke
- fire
nc: 2
path: /content/fire-smoke/dataset/data
test: test/images
test_count: 4306
train: train/images
train_count: 14122
val: val/images
val_count: 3099



## 6. Train YOLOv8

Training results are saved to `/content/fire-smoke/runs/`.

> ⚠️ **Colab session timeout:** Colab free tier disconnects after ~90 min of inactivity. Consider mounting Google Drive (next cell) to checkpoint automatically.

In [ ]:
# OPTIONAL: Mount Google Drive to persist checkpoints across sessions
# Uncomment the lines below if you want to save to Drive

# from google.colab import drive
# drive.mount('/content/drive')
# OUTPUT_PATH = "/content/drive/MyDrive/fire-smoke"
# os.makedirs(OUTPUT_PATH, exist_ok=True)
# yaml_dst = os.path.join(OUTPUT_PATH, 'data.yaml')  # re-save yaml to new path if changed
# with open(yaml_dst, 'w') as f: yaml.dump(cfg, f)

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8n (nano — fast and suitable for 2-class detection)
# Swap to yolov8s.pt / yolov8m.pt for better accuracy at cost of speed
model = YOLO("yolov8n.pt")

results = model.train(
    data=yaml_dst,
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,                          # GPU 0 (Colab T4)
    project=OUTPUT_PATH + "/runs",
    name="fire-smoke-yolov8n",
    exist_ok=True,
    save=True,
    plots=True,
    patience=10,                       # early stopping
    workers=2,                         # keep low on Colab
    cache=False,
    amp=True,                          # mixed precision — faster on T4
)

Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fire-smoke/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fire-smoke-yolov8n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, per

## 7. Validate on Test Set

In [ ]:
# Load the best checkpoint from training
best_weights = OUTPUT_PATH + "/runs/fire-smoke-yolov8n/weights/best.pt"

model_best = YOLO(best_weights)

metrics = model_best.val(
    data=yaml_dst,
    split='test',
    device=0,
    imgsz=640,
    batch=16,
)

print(f"\nmAP50      : {metrics.box.map50:.4f}")
print(f"mAP50-95   : {metrics.box.map:.4f}")
print(f"Precision  : {metrics.box.mp:.4f}")
print(f"Recall     : {metrics.box.mr:.4f}")

## 8. Visualise Predictions

In [ ]:
import glob
import random
from PIL import Image
import matplotlib.pyplot as plt

# Pick a few random test images
test_imgs = glob.glob(os.path.join(DATASET_PATH, 'test', 'images', '*.jpg'))
samples = random.sample(test_imgs, min(6, len(test_imgs)))

preds = model_best.predict(
    source=samples,
    conf=0.25,
    iou=0.45,
    device=0,
    save=False,
)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, result in zip(axes.flat, preds):
    img_bgr = result.plot()          # numpy array with boxes drawn
    img_rgb = img_bgr[:, :, ::-1]   # BGR → RGB
    ax.imshow(img_rgb)
    ax.axis('off')
plt.suptitle('Sample Predictions (fire / smoke)', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_PATH + '/sample_predictions.png', dpi=120)
plt.show()

## 9. Export Best Model

Export to ONNX for deployment or further use.

In [ ]:
# Export to ONNX
model_best.export(
    format='onnx',
    imgsz=640,
    opset=12,
    simplify=True,
)

print("\nExported files in output:")
for f in sorted(os.listdir(OUTPUT_PATH + "/runs/fire-smoke-yolov8n/weights")):
    print(" ", f)

## 10. Save Results Summary

In [ ]:
import json

summary = {
    "model": "YOLOv8n",
    "dataset": "sayedgamal99/smoke-fire-detection-yolo (D-Fire)",
    "classes": ["fire", "smoke"],
    "epochs": 30,
    "imgsz": 640,
    "mAP50": round(metrics.box.map50, 4),
    "mAP50_95": round(metrics.box.map, 4),
    "precision": round(metrics.box.mp, 4),
    "recall": round(metrics.box.mr, 4),
    "best_weights": best_weights,
}

with open(OUTPUT_PATH + "/results_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

## 11. Download Results to Your Machine

Download the trained weights and summary directly from Colab.

In [ ]:
from google.colab import files

# Download best weights
files.download(best_weights)

# Download results summary
files.download(OUTPUT_PATH + "/results_summary.json")

# Download sample predictions image
files.download(OUTPUT_PATH + "/sample_predictions.png")